# Asclepius Synthetic Clinical Notes

Downloads `starmpcc/Asclepius-Synthetic-Clinical-Notes` from Hugging Face (~158k synthetic discharge summaries with instruction–answer pairs, CC BY-NC-SA 4.0) and saves a local copy under `data/asclepius/`.

**Part 1** downloads and saves the data (run once). **Part 2** reads the saved parquet, so you can start there without re-downloading.

# Part 1: Download (run once)

In [1]:
%pip install -q datasets pandas pyarrow

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import pandas as pd
from datasets import load_dataset

DATA_DIR = Path("data/asclepius")
DATA_DIR.mkdir(parents=True, exist_ok=True)

/Users/shayan/Projects/PatientInsight/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/shayan/Projects/PatientInsight/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [3]:
ds = load_dataset("starmpcc/Asclepius-Synthetic-Clinical-Notes")
ds

Generating train split: 100%|██████████| 158114/158114 [00:03<00:00, 46590.19 examples/s]


DatasetDict({
    train: Dataset({
        features: ['patient_id', 'note', 'question', 'answer', 'task'],
        num_rows: 158114
    })
})

In [4]:
df = ds["train"].to_pandas()
print(df.shape)
print(df.columns.tolist())
df.head()

(158114, 5)
['patient_id', 'note', 'question', 'answer', 'task']


,patient_id,note,question,answer,task
0,0,Discharge Summary:\n\nPatient: 60-year-old mal...,Can you provide a simplified paraphrase of the...,The healthcare team used a gradual approach to...,Paraphrasing
1,1,Discharge Summary:\n\nAdmission Date: [Insert ...,Which coreferences were resolved in the hospit...,The hospital course section resolved the coref...,Coreference Resolution
2,2,Hospital Course Summary:\n\nAdmission Date: [I...,What were the key improvements in the patient'...,"During the hospital course, the patient's medi...",Summarization
3,3,Discharge Summary:\n\nPatient: 69-year-old mal...,What roles did physical therapists have in the...,Physical therapists were responsible for ensur...,Relation Extraction
4,4,Discharge Summary:\n\nPatient Information:\n- ...,What manual airway clearance techniques were u...,The discharge summary stated that 1-2 physical...,Relation Extraction


## Quick look

In [5]:
# Task distribution (if the column exists)
if "task" in df.columns:
    display(df["task"].value_counts())

# Note length in words
if "note" in df.columns:
    display(df["note"].str.split().str.len().describe())

task
Question Answering                 20038
Abbreviation Expansion             20015
Relation Extraction                19926
Named Entity Recognition           19848
Summarization                      19756
Temporal Information Extraction    19654
Coreference Resolution             19622
Paraphrasing                       19255
Name: count, dtype: int64

count    158114.000000
mean        277.405119
std          82.476246
min          94.000000
25%         218.000000
50%         267.000000
75%         325.000000
max         869.000000
Name: note, dtype: float64

In [6]:
# One full example
row = df.sample(1, random_state=0).iloc[0]
for col, val in row.items():
    print(f"=== {col} ===\n{val}\n")

=== patient_id ===
48489

=== note ===
DISCHARGE SUMMARY:

Patient Name: [REDACTED]

Medical Record Number: [REDACTED]

Date of Admission: [REDACTED]

Date of Discharge: [REDACTED]

Admitting Diagnosis: Subcutaneous swelling on right cheek

History of Present Illness: The patient, a 29-year-old male of African descent, presented to the Plastic and Reconstructive Surgery Outpatient Department with a slow-growing, non-tender swelling on the right cheek. An excisional biopsy was performed which revealed the diagnosis of KD. The patient defaulted from the clinic after the biopsy.

Hospital Course: The patient re-presented 2 years later with a larger swelling at the same site, and a diagnosis of recurrent KD was made on histopathological examination of the re-excised tissue. The patient refused radiation therapy and was monitored closely in the outpatient department. He has shown no signs of recurrence in the past 14 months.

Discharge Diagnosis: Recurrent KD

Condition at Discharge: Stable

## Save locally

In [7]:
out = DATA_DIR / "asclepius.parquet"
df.to_parquet(out, index=False)
print(f"Saved {len(df):,} rows to {out}")

Saved 158,114 rows to data/asclepius/asclepius.parquet


# Part 2: Inspect (start here after the first run)

Reads the local parquet saved in Part 1. No download needed.

In [5]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("data/asclepius/asclepius.parquet")
assert DATA_PATH.exists(), f"{DATA_PATH} not found. Run Part 1 once to download and save the data."
df = pd.read_parquet(DATA_PATH)
print(df.shape)

(158114, 5)


## Inspect notes and their questions

Helpers to read full notes alongside the question/answer pairs attached to them.

In [12]:
import textwrap

# Column names (adjust here if the dataset uses different ones)
ID, NOTE, Q, A, TASK = "patient_id", "note", "question", "answer", "task"
print("Columns:", df.columns.tolist())

def wrap(text, width=110):
    return "\n".join(textwrap.fill(line, width) if line.strip() else "" for line in str(text).splitlines())

def show_note(pid, max_note_chars=None, task_filter=None):
    """Print one note followed by its question/answer pairs (optionally only those whose task contains task_filter)."""
    rows = df[df[ID] == pid]
    if task_filter:
        rows = rows[rows[TASK].str.contains(task_filter, case=False, na=False)]
    note = rows.iloc[0][NOTE]
    n_words = len(str(note).split())
    print("=" * 110)
    print(f"PATIENT {pid}  |  {n_words} words  |  {len(rows)} question(s)")
    # print("=" * 110)
    # print(wrap(note if max_note_chars is None else str(note)[:max_note_chars] + " [...]"))
    for k, (_, r) in enumerate(rows.iterrows(), 1):
        print("\n" + "-" * 110)
        print(f"Q{k} [{r.get(TASK, '?')}]: {wrap(r[Q])}")
        # print(f"\nA{k}: {wrap(r[A])}")
    print()

Columns: ['patient_id', 'note', 'question', 'answer', 'task']


In [7]:
# How many questions does each note have?
q_per_note = df.groupby(ID).size()
print(q_per_note.describe())
print("\nNotes with >1 question:", (q_per_note > 1).sum())

count    158114.0
mean          1.0
std           0.0
min           1.0
25%           1.0
50%           1.0
75%           1.0
max           1.0
dtype: float64

Notes with >1 question: 0


### Patients with temporal information extraction questions

Shows only notes that have a temporal-extraction question, and only those questions.

In [13]:
temporal = df[df[TASK].str.contains("temporal", case=False, na=False)]
print("Temporal task label(s):", temporal[TASK].unique().tolist())
print(f"{len(temporal):,} temporal questions across {temporal[ID].nunique():,} notes\n")

for pid in temporal[ID].drop_duplicates().sample(10, random_state=42):
    show_note(pid, task_filter="temporal")

Temporal task label(s): ['Temporal Information Extraction']
19,654 temporal questions across 19,654 notes

PATIENT 108867  |  150 words  |  1 question(s)

--------------------------------------------------------------------------------------------------------------
Q1 [Temporal Information Extraction]: What was the patient's diagnosis that required surgical intervention, and how was the procedure carried out
according to the discharge summary?

PATIENT 166988  |  186 words  |  1 question(s)

--------------------------------------------------------------------------------------------------------------
Q1 [Temporal Information Extraction]: What was the patient's CD4 cell count and plasma viral load at the time of admission?

PATIENT 85375  |  284 words  |  1 question(s)

--------------------------------------------------------------------------------------------------------------
Q1 [Temporal Information Extraction]: What ocular abnormalities were identified in the patient's eye examinat

### One example per task type

Useful for seeing what each of the task categories actually asks.

In [14]:
for task, group in df.groupby(TASK):
    print(f"\n\n######## TASK: {task}  ({len(group):,} rows) ########")
    show_note(group.sample(1, random_state=0).iloc[0][ID], max_note_chars=1500)



######## TASK: Abbreviation Expansion  (20,015 rows) ########
PATIENT 30702  |  284 words  |  1 question(s)

--------------------------------------------------------------------------------------------------------------
Q1 [Abbreviation Expansion]: What was the abbreviation expanded as "EVD" in the discharge summary related to the patient's acute
obstructive hydrocephalus?



######## TASK: Coreference Resolution  (19,622 rows) ########
PATIENT 96168  |  208 words  |  1 question(s)

--------------------------------------------------------------------------------------------------------------
Q1 [Coreference Resolution]: What coreferents are present in the given discharge summary related to the patient's treatment with Endurant
and Valiant system for AAA and TAA?



######## TASK: Named Entity Recognition  (19,848 rows) ########
PATIENT 63631  |  295 words  |  1 question(s)

--------------------------------------------------------------------------------------------------------------


### Browse questions only (fast skim)

In [15]:
# Skim 20 questions from one task without the notes. Change the task name to explore others.
task_name = df[TASK].value_counts().index[0]
for q in df[df[TASK] == task_name][Q].sample(20, random_state=1):
    print("•", wrap(q))

• What was the patient's hospital course and eventual cause of death, given their diagnosis of shortness of
breath and dry cough associated with Kaposi's sarcoma and positive anti-HIV ELISA test?
• Was KardiaMobile 6L used during the hospital stay, and if so, what was the purpose of the evaluation?
• What was the reason for the patient's hospitalization and what procedure was performed to address the issue?
• What surgical procedures were performed and what was the diagnosis for a 44-year-old female patient admitted
with right knee pain and limited range of motion?
• What genetic variants were identified in the patient's FOXRED1 gene during Trio whole-exome sequencing and
what is the pathogenicity of the novel variant c.352C>T (p.Q118X)?
• What anesthesia procedure was performed on the patient with a fracture of the lower third of the left humerus
mentioned in the given discharge summary?
• What tests and management strategies were implemented for a patient with dengue fever and rash a

### Search notes by keyword

In [ ]:
def search_notes(keyword, n=3):
    hits = df[df[NOTE].str.contains(keyword, case=False, na=False)][ID].drop_duplicates()
    print(f"{len(hits):,} notes mention '{keyword}'. Showing {min(n, len(hits))}.\n")
    for pid in hits.head(n):
        show_note(pid, max_note_chars=2000)

search_notes("heart failure", n=2)